# RapidFire AI RAG Experiment - Documentation Q&A Chatbot

This notebook runs a RAG-based evaluation pipeline on the RapidFire AI documentation using a fixed chunking configuration.

## Load API Key

In [ ]:
from pathlib import Path
import os

# Get API Key
TRITON_API_KEY = Path("~/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

# Set environment variables
os.environ.setdefault("OPENAI_API_KEY", TRITON_API_KEY)
os.environ.setdefault("JUDGE_BASE_URL", "https://tritonai-api.ucsd.edu/v1")
os.environ.setdefault("JUDGE_MODEL", "claude-sonnet-4-6-aws")

## Import Required Libraries

In [ ]:
# RapidFireAI Imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

# Standard library imports
import re
import json
from typing import List as listtype, Dict, Any, Optional
from pathlib import Path
import tiktoken

# Data and ML imports
import pandas as pd
from datasets import Dataset

# LangChain imports
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_openai import OpenAIEmbeddings

# Import evaluation functions
from project1_eval import call_judge, f1_at_k, precision_at_k, recall_at_k, to_spans
from rapidfire_integration_example import sample_compute_metrics_fn, sample_accumulate_metrics_fn

## Load Dataset

Load the validation set golden Q&A pairs and experiment.json file.

In [ ]:
# Load input JSON file
input_file = "validation-set-golden-qa-pairs.json"
output_file = "experiment_output.json"

with open(input_file, "r") as f:
    data = json.load(f)

# Build dataset rows
rows = [
    {
        "query_id": int(entry["question_id"]),          
        "query": str(entry["question"]),
        "reference_answer": str(entry.get("reference_answer", "")), 
        "source_evidence": entry.get("source_evidence", []),
    }
    for entry in data
]

dataset = Dataset.from_list(rows)
print(f"Loaded {len(dataset)} examples from {input_file}")
print(f"Dataset preview:")
print(dataset[0])

## Create Experiment

In [ ]:
experiment = Experiment(experiment_name="experiment_exploration_fixed_chunking", mode="evals")

## Define RAG Configuration

Configure the LangChain RAG pipeline with fixed chunk size=128 and overlap=8.

In [ ]:
# Chunking parameters - test both 128 and 512
CHUNK_SIZES = [128, 512]
CHUNK_OVERLAP = 8
batch_size = 32

def build_rag_for_chunking(chunk_size: int, chunk_overlap: int) -> RFLangChainRagSpec:
    """Build a RAG configuration for a given chunk size and overlap."""
    return RFLangChainRagSpec(
        document_loader=DirectoryLoader(
            path="sourcedocs/sourcedocs/",
            glob="**/*.rst",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
            sample_seed=1337,
        ),
        text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            add_start_index=True,
        ),
        embedding_cfg={
            "class": OpenAIEmbeddings,
            "model": "api-tgpt-embeddings",
            "api_key": TRITON_API_KEY,
            "base_url": "https://tritonai-api.ucsd.edu",
            "check_embedding_ctx_length": False,
        },
        vector_store_cfg={"type": "faiss", "batch_size": batch_size},
        search_cfg=List([
            {"type": "similarity", "k": 10},
            {"type": "similarity", "k": 20}
        ]),
        reranker_cfg=List([
            {
                "class": CrossEncoderReranker,
                "model_name": "BAAI/bge-reranker-v2-m3",
                "model_kwargs": {"device": "cpu"},
                "top_n": List(2, 5),
            }
        ]),
        enable_gpu_search=False,
    )

print(f"RAG configuration function created for chunk_sizes={CHUNK_SIZES}, chunk_overlap={CHUNK_OVERLAP}")

## Define Instructions and Helper Functions

In [ ]:
INSTRUCTIONS = """You are a precise technical assistant for the RapidFire AI documentation.
You will be given a user question and relevant context chunks retrieved from the RapidFire AI docs.

Rules:
- Answer using ONLY information present in the provided context. Do not use outside knowledge.
- Be specific and complete — include parameter names, types, defaults, and exact values when present.
- For procedural questions, list the steps in order.
- For comparative questions, clearly distinguish between the two things being compared.
- For factual/lookup questions, give the exact answer directly.
- If the context does not contain enough information to answer, say: "The provided context does not contain enough information to answer this question."
- Do not add caveats, filler phrases, or unnecessary preamble. Get to the answer immediately.
- Stay within 2000 tokens total context budget.

Respond with your answer only. No reasoning prefix needed.
Question: "What are the two main execution functions provided by the Experiment class for launching workflows?"
Answer: "The two main execution functions are run_fit() for training/evaluation workflows and run_evals() for LLM evaluation workflows."
Source Evidence: source_evidence": [
    { "file": "experiment.rst", "lines": [69, 75] },
    { "file": "experiment.rst", "lines": [154, 160] }
    ]
"""

# Token-aware truncation
MAX_TOKENS_PER_QUERY: int = 2000
SAFETY_TOKENS: int = 50

def _truncate_context(text: str, max_chars: Optional[int]) -> str:
    if max_chars is None or len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n\n[context truncated]"

def _truncate_context_by_tokens(text: str, max_tokens: Optional[int], encoding) -> str:
    if max_tokens is None:
        return text
    if max_tokens <= 0:
        return ""
    if encoding is None:
        return _truncate_context(text, max_chars=int(max_tokens * 4))
    toks = encoding.encode(text)
    if len(toks) <= max_tokens:
        return text
    try:
        return encoding.decode(toks[:max_tokens]) + "\n\n[context truncated]"
    except Exception:
        try:
            return "".join(toks[:max_tokens]) + "\n\n[context truncated]"
        except Exception:
            return "\n\n[context truncated]"

def chunk_to_lines(doc: Document) -> listtype:
    """Convert a chunk's character `start_index` into [start_line, end_line]."""
    src = doc.metadata["source"]
    start_idx = doc.metadata["start_index"]
    text = Path(src).read_text(encoding="utf-8")
    start_line = text[:start_idx].count("\n") + 1
    end_line = start_line + (doc.page_content or "").count("\n")
    if end_line < start_line:
        end_line = start_line
    return [start_line, end_line]

## Define Preprocessing and Postprocessing Functions

In [ ]:
# Initialize output storage
output_rows = []
output_rows_jsonl = Path(output_file + ".rows.jsonl")
if output_rows_jsonl.exists():
    output_rows_jsonl.unlink()

def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)
    serialized_context = rag.serialize_documents(all_context)
    
    # Token-aware per-query truncation
    encoding = tiktoken.get_encoding("gpt2")
    system_tokens = len(encoding.encode(INSTRUCTIONS or ""))
    template_tokens = len(encoding.encode("\nQuestion:\n\nContext:\n\nAnswer:"))
    new_serialized = []
    
    for question, ctx in zip(batch.get("query", []), serialized_context):
        q_tokens = len(encoding.encode(question or ""))
        avail = MAX_TOKENS_PER_QUERY - (system_tokens + q_tokens + template_tokens + SAFETY_TOKENS)
        if avail <= 0:
            new_serialized.append("")
        else:
            new_serialized.append(_truncate_context_by_tokens(ctx, avail, encoding))
    
    serialized_context = new_serialized
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    batch["ground_truth_spans"] = [
        [
            (item["file"], int(item["lines"][0]), int(item["lines"][1]))
            for item in evidence
            if item.get("file") and item.get("lines") and len(item["lines"]) >= 2
        ]
        for evidence in batch.get("source_evidence", [])
    ]

    per_doc_lines = [[chunk_to_lines(doc) for doc in docs] for docs in all_context]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"\nQuestion:\n{question}\n\nContext:\n{context}\n\nAnswer:"
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "serialized_context": serialized_context,
        "retrieved_context": serialized_context,
        "sources": [
            [
                {"file": Path(doc.metadata["source"]).name, "lines": lines}
                for doc, lines in zip(docs, doc_lines)
            ]
            for docs, doc_lines in zip(all_context, per_doc_lines)
        ],
        "retrieved_spans": [
            [
                (Path(doc.metadata["source"]).name, lines[0], lines[1])
                for doc, lines in zip(docs, doc_lines)
            ]
            for docs, doc_lines in zip(all_context, per_doc_lines)
        ],
        **batch,
    }

def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Postprocess outputs produced by generator model"""
    batch["answer"] = batch["generated_text"]

    for qid, ans, ctx, srcs in zip(
        batch["query_id"],
        batch["answer"],
        batch["retrieved_context"],
        batch["sources"],
    ):
        row = {
            "question_id": int(qid),
            "answer": ans,
            "retrieved_context": ctx,
            "sources": srcs,
        }
        output_rows.append(row)
        # Persist each row so outputs survive multi-process Ray workers
        with open(output_rows_jsonl, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    return batch

## Define Generator Configuration

In [ ]:
# Create OpenAI generator configs for each chunk size
openai_configs = []
for chunk_size in CHUNK_SIZES:
    openai_configs.append(
        RFOpenAIAPIModelConfig(
            client_config={
                "api_key": TRITON_API_KEY,
                "base_url": "https://tritonai-api.ucsd.edu",
                "max_retries": 2
            },
            model_config={
                "model": "api-mistral-small-3.2-2506",
                "max_completion_tokens": 2048,
            },
            rpm_limit=120,
            tpm_limit=1_000_000,
            rag=build_rag_for_chunking(chunk_size=chunk_size, chunk_overlap=CHUNK_OVERLAP),
            prompt_manager=None,
        )
    )

# Create configuration set
config_set = {
    "openai_config": List(openai_configs),
    "batch_size": batch_size,
    "preprocess_fn": openai_sample_preprocess_fn,
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
}

config_group = RFGridSearch(config_set)

## Run Evaluation

In [ ]:
# Launch evals
results = experiment.run_evals(
    config_group=config_group,
    dataset=dataset,
    num_shards=4,
    num_actors=4,
    seed=42,
)

print("Evaluation completed!")

## Save Output

In [ ]:
# Write final output JSON
output_rows.sort(key=lambda x: x["question_id"])

if output_rows_jsonl.exists():
    with open(output_rows_jsonl, "r", encoding="utf-8") as f:
        output_rows = [json.loads(line) for line in f if line.strip()]
    output_rows.sort(key=lambda x: x["question_id"])

with open(output_file, "w") as f:
    json.dump(output_rows, f, indent=2)

print(f"Output saved to {output_file}")
print(f"Total examples processed: {len(output_rows)}")

## End Experiment

In [ ]:
experiment.end()
print("Experiment ended.")

## View Experiment Logs

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")